In [ ]:
import numpy as np
import pandas as pd
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

# 查看非灾难推文的前5条
non_disaster_tweets = train_df[train_df["target"] == 0]["text"]
print("非灾难推文示例：")
print(non_disaster_tweets.head())

# 查看灾难推文的前5条
disaster_tweets = train_df[train_df["target"] == 1]["text"]
print("\n灾难推文示例：")
print(disaster_tweets.head())

In [ ]:
# 数据预处理：简单的文本清理
def preprocess_text(text):
    if isinstance(text, str):
        # 移除URLs、用户提及和标点符号
        text = text.lower()
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'[^\w\s]', '', text)
        return text
    return text

import re
train_df['cleaned_text'] = train_df['text'].apply(preprocess_text)
test_df['cleaned_text'] = test_df['text'].apply(preprocess_text)

In [ ]:
# TF-IDF向量化（带参数调优）
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),  # 包含unigram和bigram
    stop_words='english',
    min_df=2,
    max_df=0.8
)

train_vectors = tfidf_vectorizer.fit_transform(train_df['cleaned_text'])
test_vectors = tfidf_vectorizer.transform(test_df['cleaned_text'])

转化后的向量 用三种模型分类

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# 1. Logistic Regression（逻辑回归）
print("训练Logistic Regression...")
lr_clf = LogisticRegression(
    C=1.0,
    solver='liblinear',
    random_state=42,
    max_iter=1000
)
lr_clf.fit(train_vectors, train_df["target"])

# 2. Linear SVC（支持向量机）
print("训练Linear SVC...")
svc_clf = LinearSVC(
    C=0.5,
    random_state=42,
    max_iter=1000
)
svc_clf.fit(train_vectors, train_df["target"])

# 3. 随机森林（适合集成学习）
print("训练Random Forest...")
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(train_vectors, train_df["target"])

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# 交叉验证评估
print("\n模型交叉验证得分：")
models = {
    'LogisticRegression': lr_clf,
    'LinearSVC': svc_clf,
    'RandomForest': rf_clf
}

for name, model in models.items():
    scores = cross_val_score(model, train_vectors, train_df["target"], cv=5, scoring='f1')
    print(f"{name}: F1得分 = {scores.mean():.4f} (±{scores.std():.4f})")

LinearSVC 支持向量机分数最高，用它预测最后结果

In [ ]:
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sample_submission["target"] = svc_clf.predict(test_vectors)
sample_submission.to_csv("submission.csv", index=False)